# 1. Exploratory Data Analysis

## The Data

| Column Name | Description                                                                                       |
|-------------|---------------------------------------------------------------------------------------------------|
| caseid      | Integer ID of the respondent.                                                                     |
| pregordr    | Pregnancy serial number (1 = first pregnancy, 2 = second, etc.).                                  |
| prglngth    | Duration of the pregnancy in weeks.                                                               |
| outcome     | Outcome code of the pregnancy (1 = live birth).                                                   |
| birthord    | Serial number for live births (1 = first child, 2 = second, etc.). Blank for non-live births.     |
| birthwgt_lb | Pounds part of the baby’s birth weight.                                                           |
| birthwgt_oz | Ounces part of the baby’s birth weight.                                                           |
| agepreg     | Mother’s age at the end of the pregnancy.                                                         |
| finalwgt    | Statistical weight representing how many people in the U.S. population the respondent represents. |

## Glossary

| Term                  | Definition                                                                                                              |
|-----------------------|-------------------------------------------------------------------------------------------------------------------------|
| Anecdotal evidence    | Data collected informally from a small number of individual cases, often without systematic sampling.                   |
| Cross-sectional study | A study that collects data from a representative sample of a population at a single point or interval in time.          |
| Cycle                 | One data-collection interval in a study that collects data at multiple intervals in time.                               |
| Population            | The entire group of individuals or items that is the subject of a study.                                                |
| Sample                | A subset of a population, often chosen at random.                                                                       |
| Respondents           | People who participate in a survey and respond to questions.                                                            |
| Representative        | A sample is representative if it is similar to the population in ways that are important for the purposes of the study. |
| Stratified            | A sample is stratified if it deliberately oversamples some groups to ensure valid conclusions.                          |
| Oversampled           | A group is oversampled if its members have a higher chance of appearing in a sample.                                    |
| Variable              | A collection of responses or computed values in a dataset.                                                              |
| Codebook              | A document describing the variables and structure of a dataset.                                                         |
| Recode                | A variable computed from other variables in the dataset.                                                                |
| Raw data              | Data that has not been processed after collection.                                                                      |
| Data cleaning         | The process of correcting errors, handling missing values, and preparing data for analysis.                             |
| Statistic             | A value that summarizes a property of a sample.                                                                         |
| Standard deviation    | A statistic that measures how spread out data is around the mean.                                                       |


https://allendowney.github.io/ThinkStats/chap01.html

In [8]:
import pandas as pd
from statadict import parse_stata_dict
import numpy as np

# The following function takes these file names as arguments,
# reads the dictionary, and uses the results to read the data file.
def read_stata(dct_file, dat_file):
    stata_dict = parse_stata_dict(dct_file)
    resp = pd.read_fwf(
        dat_file,
        names=stata_dict.names,
        colspecs=stata_dict.colspecs,
        compression="gzip",
    )
    return resp

# The data is stored in two files, a "dictionary" that
# describes the format of the data, and a data file.
dct_file_2002FemPreg = "../data/2002FemPreg.dct"
dat_file_2002FemPreg = "../data/2002FemPreg.dat.gz"

preg = read_stata(dct_file_2002FemPreg, dat_file_2002FemPreg)

In [2]:
# The DataFrame has an attribute called shape
# that contains the number of rows and columns.
preg.shape

(13593, 243)

In [3]:
preg.columns

Index(['caseid', 'pregordr', 'howpreg_n', 'howpreg_p', 'moscurrp', 'nowprgdk',
       'pregend1', 'pregend2', 'nbrnaliv', 'multbrth',
       ...
       'poverty_i', 'laborfor_i', 'religion_i', 'metro_i', 'basewgt',
       'adj_mod_basewgt', 'finalwgt', 'secu_p', 'sest', 'cmintvw'],
      dtype='str', length=243)

To check these totals, we’ll use the value_counts method, which counts the number of times each value appears, and sort_index, which sorts the results according to the values in the Index (the left column).

In [4]:
preg["outcome"].value_counts().sort_index()

outcome
1    9148
2    1862
3     120
4    1921
5     190
6     352
Name: count, dtype: int64

In [5]:
counts = preg["birthwgt_lb"].value_counts(dropna=False).sort_index()
counts

birthwgt_lb
0.0        8
1.0       40
2.0       53
3.0       98
4.0      229
5.0      697
6.0     2223
7.0     3049
8.0     1889
9.0      623
10.0     132
11.0      26
12.0      10
13.0       3
14.0       3
15.0       1
51.0       1
97.0       1
98.0       1
99.0      57
NaN     4449
Name: count, dtype: int64

The counts for 6, 7, and 8 pounds are consistent with the codebook. To check the counts for the weight range from 0 to 5 pounds, we can use an attribute called loc – which is short for “location” – and a slice index to select a subset of the counts.

In [9]:
counts.loc[0:5]

birthwgt_lb
0.0      8
1.0     40
2.0     53
3.0     98
4.0    229
5.0    697
Name: count, dtype: int64

## Data cleaning

In [11]:
preg["birthwgt_lb"] = preg["birthwgt_lb"].replace([51, 97, 98, 99], np.nan)

## Transformation
As another kind of data cleaning, sometimes we have to convert data into different formats, and perform other calculations.

In [12]:
preg["agepreg"].mean()

np.float64(2468.8151197039497)

In [13]:
preg["agepreg"] /= 100.0
preg["agepreg"].mean()

np.float64(24.6881511970395)

As another example, birthwgt_lb and birthwgt_oz contain birth weights with the pounds and ounces in separate columns. It will be more convenient to combine them into as single column that contains weights in pounds and fractions of a pound.

In [14]:
preg["birthwgt_oz"].value_counts(dropna=False).sort_index()

birthwgt_oz
0.0     1037
1.0      408
2.0      603
3.0      533
4.0      525
5.0      535
6.0      709
7.0      501
8.0      756
9.0      505
10.0     475
11.0     557
12.0     555
13.0     487
14.0     475
15.0     378
97.0       1
98.0       1
99.0      46
NaN     4506
Name: count, dtype: int64

In [ ]:
preg["birthwgt_oz"] = preg["birthwgt_oz"].replace([97, 98, 99], np.nan)

In [16]:
preg["totalwgt_lb"] = preg["birthwgt_lb"] + preg["birthwgt_oz"] / 16.0
preg["totalwgt_lb"].mean()

np.float64(7.293634412153237)

In [18]:
weights = preg["totalwgt_lb"]
n = weights.count()
n

np.int64(9084)

Variance is a statistic that quantifies the spread of a set of values. It is the mean of the squared deviations, which are the distances of each point from the mean.

In [21]:
mean = weights.mean()
squared_deviations = (weights - mean) ** 2

In [23]:
var = squared_deviations.sum() / n
var

np.float64(2.1391278316141302)

In [22]:
weights.var(ddof=0)

np.float64(2.1391278316141302)

A better option is the standard deviation, which is the square root of variance

In [24]:
std = np.sqrt(var)
std

np.float64(1.4625757524361362)

In [25]:
weights.std(ddof=0)

np.float64(1.4625757524361362)

Informally, values that are one or two standard deviations from the mean are common – values farther from the mean are rare.

In [26]:
subset = preg.query("caseid == 10229")
subset.shape

(7, 244)

In [27]:
subset["outcome"].values

array([4, 4, 4, 4, 4, 4, 1])

## Exercises Chapter 1

- Select the birthord column from preg, print the value counts, and compare to results published in the codebook at https://ftp.cdc.gov/pub/Health_Statistics/NCHS/Dataset_Documentation/NSFG/Cycle6Codebook-Pregnancy.pdf.

In [28]:
preg['birthord'].value_counts()

birthord
1.0     4413
2.0     2874
3.0     1234
4.0      421
5.0      126
6.0       50
7.0       20
8.0        7
9.0        2
10.0       1
Name: count, dtype: int64

- Create a new column named totalwgt_kg that contains birth weight in kilograms (there are approximately 2.2 pounds per kilogram). Compute the mean and standard deviation of the new column.

In [32]:
preg['totalwgt_kg'] = preg['totalwgt_lb'] / 2.2
preg['totalwgt_kg'].mean()

np.float64(3.315288369160562)

- What are the pregnancy lengths for the respondent with caseid 2298?

In [33]:
length = preg.query('caseid == 2298')
length['prglngth'].values

array([40, 36, 30, 40])

- What was the birth weight of the first baby born to the respondent with caseid 5013?

In [41]:
weight = preg.query('caseid == 5013 and pregordr == 1')
weight['totalwgt_kg'].values

array([3.35227273])